In [ ]:

import jax
import jax.numpy as jnp
from jax import lax, jit, vmap, pmap
from functools import partial
import time

# --- Configuration
MAX_RECURSION_DEPTH    = 1_000_000   # Maximum recursion depth.
OPTIMAL_DEPTH_STEP     = 250_000     # Base branch step length.
DIMENSIONAL_CONSTRAINT = 0.8
BATCH_SIZE             = 1_000_000   # Reduced batch size for testing (scale up later)
NUM_DEVICES            = jax.local_device_count()  # e.g. 8 cores
LOCAL_BATCH_SIZE       = BATCH_SIZE // NUM_DEVICES

VAL_CLAMP_LOW  = -100.0
VAL_CLAMP_HIGH =  100.0

# -------------------------------------------------------------------------
# 1) Dynamic pi & phi functions (with modified decay)
# -------------------------------------------------------------------------
@jit
def dynamic_pi(depth, scale_factor):
    depth = jnp.minimum(depth, MAX_RECURSION_DEPTH)
    return jnp.pi * jnp.log1p(depth + 1) * scale_factor * DIMENSIONAL_CONSTRAINT

@jit
def dynamic_phi(depth, scale_factor):
    depth = jnp.minimum(depth, MAX_RECURSION_DEPTH)
    return (1 + jnp.sqrt(5)) / 2 * jnp.exp(-depth / ((scale_factor * 10) + 1)) * DIMENSIONAL_CONSTRAINT

@jit
def stabilize_depth(depth):
    return depth / (1 + jnp.log1p(depth + 1))

# -------------------------------------------------------------------------
# 2) Base function: 250K-step linear recursion.
# -------------------------------------------------------------------------
@partial(jit, static_argnames=["depth"])
def dppu_with_dynamic_pi_phi(x, depth=OPTIMAL_DEPTH_STEP, scale_factor=1.0):
    depth = stabilize_depth(jnp.minimum(depth, MAX_RECURSION_DEPTH))
    def body_fn(i, val):
        pi_dyn  = dynamic_pi(i, scale_factor)
        phi_dyn = dynamic_phi(i, scale_factor)
        scale   = jnp.log1p(i + 1) * scale_factor * DIMENSIONAL_CONSTRAINT
        safe_val = jnp.clip(val, VAL_CLAMP_LOW, VAL_CLAMP_HIGH)
        new_val = jnp.sin(safe_val * scale * pi_dyn) * jnp.exp(-safe_val / (phi_dyn + 1))
        return new_val
    return lax.fori_loop(0, depth.astype(jnp.int32), body_fn, x)

# -------------------------------------------------------------------------
# 3) Process with Branch Recycling fused with lax.scan.
#    Each device works on its shard (of shape [LOCAL_BATCH_SIZE]) and
#    iterates a given number of times.
# -------------------------------------------------------------------------
def process_with_branches_scan(x, iterations, num_branches=2, branch_depth=OPTIMAL_DEPTH_STEP, scale_factor=1.0):
    def body_fn(carry, _):
        # Replicate the local shard to create branches: shape (num_branches, LOCAL_BATCH_SIZE)
        xs = jnp.stack([carry] * num_branches, axis=0)
        branch_fn = vmap(lambda xi: dppu_with_dynamic_pi_phi(xi, depth=branch_depth, scale_factor=scale_factor))
        branch_outputs = branch_fn(xs)
        new_val = jnp.sum(branch_outputs, axis=0)
        return new_val, new_val  # (new carry, auxiliary output if needed)
    # Use jnp.arange(iterations) so that xs has a leading axis.
    final_val, intermediates = lax.scan(body_fn, x, jnp.arange(iterations))
    return final_val, intermediates

# -------------------------------------------------------------------------
# 4) Wrap the function with pmap, marking static parameters.
# -------------------------------------------------------------------------
p_process_with_branches_scan = pmap(
    process_with_branches_scan,
    in_axes=(0, None, None, None, None),
    static_broadcasted_argnums=(1, 2, 3, 4)  # iterations, num_branches, branch_depth, scale_factor
)

# -------------------------------------------------------------------------
# 5) Benchmarking function using pmap.
# -------------------------------------------------------------------------
def run_benchmarks_pmap(batch_input, depths=(250_000, 500_000, 1_000_000),
                        num_branches=2, branch_depth=OPTIMAL_DEPTH_STEP,
                        scale_factor=1.0, num_trials=2):
    results = []
    for depth in depths:
        iterations = depth // branch_depth  # This is a Python int.
        print(f"\n--- Running benchmark for total_depth={depth} (iterations={iterations}) ---")
        # Split the input batch into shards for each device.
        sharded_input = batch_input.reshape((NUM_DEVICES, -1))
        for trial in range(num_trials):
            start_time = time.time()
            final_shard, _ = p_process_with_branches_scan(
                sharded_input, iterations, num_branches, branch_depth, scale_factor
            )
            # Reassemble results from devices.
            final_x = jax.device_get(final_shard).reshape(-1)
            jax.block_until_ready(final_x)
            elapsed = time.time() - start_time
            mean_val = float(jnp.mean(final_x))
            results.append({"depth": depth, "trial": trial, "time": elapsed, "mean_output": mean_val})
            print(f"Trial {trial}: Final mean(x)={mean_val:.6f}, total time: {elapsed:.6f} sec")
    return results

# -------------------------------------------------------------------------
# 6) Main Script.
# -------------------------------------------------------------------------
if __name__ == "__main__":
    # For testing, we start with a batch of 1M values; later, try scaling to 50M.
    batch_input = jnp.linspace(0, 10, BATCH_SIZE)

    bench_results = run_benchmarks_pmap(
        batch_input,
        depths=[250_000, 500_000, 1_000_000],
        num_branches=2,
        branch_depth=OPTIMAL_DEPTH_STEP,
        scale_factor=1.0,
        num_trials=2
    )

    print("\nCollected Benchmark Results:")
    for r in bench_results:
        print(r)


--- Running benchmark for total_depth=250000 (iterations=1) ---
Trial 0: Final mean(x)=0.000000, total time: 3.779658 sec
Trial 1: Final mean(x)=0.000000, total time: 2.702704 sec

--- Running benchmark for total_depth=500000 (iterations=2) ---
Trial 0: Final mean(x)=0.000000, total time: 5.849709 sec
Trial 1: Final mean(x)=0.000000, total time: 5.380418 sec

--- Running benchmark for total_depth=1000000 (iterations=4) ---
Trial 0: Final mean(x)=0.000000, total time: 11.208317 sec
Trial 1: Final mean(x)=0.000000, total time: 10.738934 sec

Collected Benchmark Results:
{'depth': 250000, 'trial': 0, 'time': 3.7796576023101807, 'mean_output': 0.0}
{'depth': 250000, 'trial': 1, 'time': 2.702704429626465, 'mean_output': 0.0}
{'depth': 500000, 'trial': 0, 'time': 5.849708795547485, 'mean_output': 0.0}
{'depth': 500000, 'trial': 1, 'time': 5.380418300628662, 'mean_output': 0.0}
{'depth': 1000000, 'trial': 0, 'time': 11.208316802978516, 'mean_output': 0.0}
{'depth': 1000000, 'trial': 1, 'time

In [ ]:
import jax
import jax.numpy as jnp
from jax import lax, jit, vmap, pmap
from functools import partial
import time

# --- Configuration
MAX_RECURSION_DEPTH    = 1_000_000   # Maximum recursion depth.
OPTIMAL_DEPTH_STEP     = 250_000     # Base branch step length.
DIMENSIONAL_CONSTRAINT = 0.8
BATCH_SIZE             = 50_000_000  # 50M samples
NUM_DEVICES            = jax.local_device_count()  # e.g. 8 cores
LOCAL_BATCH_SIZE       = BATCH_SIZE // NUM_DEVICES

VAL_CLAMP_LOW  = -100.0
VAL_CLAMP_HIGH =  100.0

# -------------------------------------------------------------------------
# 1) Dynamic pi & phi functions (with modified decay)
# -------------------------------------------------------------------------
@jit
def dynamic_pi(depth, scale_factor):
    depth = jnp.minimum(depth, MAX_RECURSION_DEPTH)
    return jnp.pi * jnp.log1p(depth + 1) * scale_factor * DIMENSIONAL_CONSTRAINT

@jit
def dynamic_phi(depth, scale_factor):
    depth = jnp.minimum(depth, MAX_RECURSION_DEPTH)
    return (1 + jnp.sqrt(5)) / 2 * jnp.exp(-depth / ((scale_factor * 10) + 1)) * DIMENSIONAL_CONSTRAINT

@jit
def stabilize_depth(depth):
    return depth / (1 + jnp.log1p(depth + 1))

# -------------------------------------------------------------------------
# 2) Base function: 250K-step linear recursion.
# -------------------------------------------------------------------------
@partial(jit, static_argnames=["depth"])
def dppu_with_dynamic_pi_phi(x, depth=OPTIMAL_DEPTH_STEP, scale_factor=1.0):
    depth = stabilize_depth(jnp.minimum(depth, MAX_RECURSION_DEPTH))
    def body_fn(i, val):
        pi_dyn  = dynamic_pi(i, scale_factor)
        phi_dyn = dynamic_phi(i, scale_factor)
        scale   = jnp.log1p(i + 1) * scale_factor * DIMENSIONAL_CONSTRAINT
        safe_val = jnp.clip(val, VAL_CLAMP_LOW, VAL_CLAMP_HIGH)
        new_val = jnp.sin(safe_val * scale * pi_dyn) * jnp.exp(-safe_val / (phi_dyn + 1))
        return new_val
    return lax.fori_loop(0, depth.astype(jnp.int32), body_fn, x)

# -------------------------------------------------------------------------
# 3) Process with Branch Recycling fused with lax.scan.
#    Each device processes its local shard (of shape [LOCAL_BATCH_SIZE])
#    through a number of iterations.
# -------------------------------------------------------------------------
def process_with_branches_scan(x, iterations, num_branches=2, branch_depth=OPTIMAL_DEPTH_STEP, scale_factor=1.0):
    def body_fn(carry, _):
        # Replicate the local shard into branches: shape (num_branches, LOCAL_BATCH_SIZE)
        xs = jnp.stack([carry] * num_branches, axis=0)
        branch_fn = vmap(lambda xi: dppu_with_dynamic_pi_phi(xi, depth=branch_depth, scale_factor=scale_factor))
        branch_outputs = branch_fn(xs)
        new_val = jnp.sum(branch_outputs, axis=0)
        return new_val, new_val  # (new carry, auxiliary output if needed)
    final_val, intermediates = lax.scan(body_fn, x, jnp.arange(iterations))
    return final_val, intermediates

# -------------------------------------------------------------------------
# 4) Wrap the function with pmap, marking static parameters.
# -------------------------------------------------------------------------
p_process_with_branches_scan = pmap(
    process_with_branches_scan,
    in_axes=(0, None, None, None, None),
    static_broadcasted_argnums=(1, 2, 3, 4)  # iterations, num_branches, branch_depth, scale_factor
)

# -------------------------------------------------------------------------
# 5) Benchmarking function using pmap.
# -------------------------------------------------------------------------
def run_benchmarks_pmap(batch_input, depths=(1_000_000,), num_branches=2, branch_depth=OPTIMAL_DEPTH_STEP,
                        scale_factor=1.0, num_trials=1):
    results = []
    for depth in depths:
        iterations = depth // branch_depth  # For 1M depth, iterations = 4
        print(f"\n--- Running benchmark for total_depth={depth} (iterations={iterations}) ---")
        # Split the input batch into shards for each device.
        sharded_input = batch_input.reshape((NUM_DEVICES, -1))
        for trial in range(num_trials):
            start_time = time.time()
            final_shard, _ = p_process_with_branches_scan(
                sharded_input, iterations, num_branches, branch_depth, scale_factor
            )
            # Reassemble results from devices.
            final_x = jax.device_get(final_shard).reshape(-1)
            jax.block_until_ready(final_x)
            elapsed = time.time() - start_time
            mean_val = float(jnp.mean(final_x))
            results.append({"depth": depth, "trial": trial, "time": elapsed, "mean_output": mean_val})
            print(f"Trial {trial}: Final mean(x)={mean_val:.6f}, total time: {elapsed:.6f} sec")
    return results

# -------------------------------------------------------------------------
# 6) Main Script.
# -------------------------------------------------------------------------
if __name__ == "__main__":
    # Create a batch of 50M samples.
    batch_input = jnp.linspace(0, 10, BATCH_SIZE)

    bench_results = run_benchmarks_pmap(
        batch_input,
        depths=[1_000_000],  # Total depth of 1M, so 4 iterations.
        num_branches=2,
        branch_depth=OPTIMAL_DEPTH_STEP,
        scale_factor=1.0,
        num_trials=1  # Adjust as needed.
    )

    print("\nCollected Benchmark Results:")
    for r in bench_results:
        print(r)


--- Running benchmark for total_depth=1000000 (iterations=4) ---
Trial 0: Final mean(x)=0.000000, total time: 525.900441 sec

Collected Benchmark Results:
{'depth': 1000000, 'trial': 0, 'time': 525.9004411697388, 'mean_output': 0.0}


In [ ]:
import jax
import jax.numpy as jnp
from jax import lax, jit, vmap, pmap
from functools import partial
import time

# --- Configuration
MAX_RECURSION_DEPTH    = 1_000_000   # Maximum recursion depth.
OPTIMAL_DEPTH_STEP     = 250_000     # Base branch step length.
DIMENSIONAL_CONSTRAINT = 0.8
BATCH_SIZE             = 50_000_000  # 50M samples
NUM_DEVICES            = jax.local_device_count()  # e.g. 8 cores
LOCAL_BATCH_SIZE       = BATCH_SIZE // NUM_DEVICES

VAL_CLAMP_LOW  = -100.0
VAL_CLAMP_HIGH =  100.0

# -------------------------------------------------------------------------
# 1) Dynamic pi & phi functions (with reduced decay)
# -------------------------------------------------------------------------
@jit
def dynamic_pi(depth, scale_factor):
    depth = jnp.minimum(depth, MAX_RECURSION_DEPTH)
    return jnp.pi * jnp.log1p(depth + 1) * scale_factor * DIMENSIONAL_CONSTRAINT

@jit
def dynamic_phi(depth, scale_factor):
    depth = jnp.minimum(depth, MAX_RECURSION_DEPTH)
    # Increase the denominator factor (scale_factor*20 instead of 10) to slow decay.
    return (1 + jnp.sqrt(5)) / 2 * jnp.exp(-depth / ((scale_factor * 20) + 1)) * DIMENSIONAL_CONSTRAINT

@jit
def stabilize_depth(depth):
    return depth / (1 + jnp.log1p(depth + 1))

# -------------------------------------------------------------------------
# 2) Base function: 250K-step linear recursion with reduced damping.
# -------------------------------------------------------------------------
@partial(jit, static_argnames=["depth"])
def dppu_with_dynamic_pi_phi(x, depth=OPTIMAL_DEPTH_STEP, scale_factor=1.0):
    depth = stabilize_depth(jnp.minimum(depth, MAX_RECURSION_DEPTH))
    def body_fn(i, val):
        pi_dyn  = dynamic_pi(i, scale_factor)
        phi_dyn = dynamic_phi(i, scale_factor)
        scale   = jnp.log1p(i + 1) * scale_factor * DIMENSIONAL_CONSTRAINT
        # Clamp value to avoid overflow/NaNs.
        safe_val = jnp.clip(val, VAL_CLAMP_LOW, VAL_CLAMP_HIGH)
        # Reduce damping by using a larger constant in the denominator.
        new_val = jnp.sin(safe_val * scale * pi_dyn) * jnp.exp(-safe_val / (phi_dyn + 10))
        return new_val
    return lax.fori_loop(0, depth.astype(jnp.int32), body_fn, x)

# -------------------------------------------------------------------------
# 3) Process with Branch Recycling fused with lax.scan.
#    Each device processes its local shard (of shape [LOCAL_BATCH_SIZE])
#    through a number of iterations.
# -------------------------------------------------------------------------
def process_with_branches_scan(x, iterations, num_branches=2, branch_depth=OPTIMAL_DEPTH_STEP, scale_factor=1.0):
    def body_fn(carry, _):
        # Replicate the local shard into branches: shape (num_branches, LOCAL_BATCH_SIZE)
        xs = jnp.stack([carry] * num_branches, axis=0)
        branch_fn = vmap(lambda xi: dppu_with_dynamic_pi_phi(xi, depth=branch_depth, scale_factor=scale_factor))
        branch_outputs = branch_fn(xs)
        new_val = jnp.sum(branch_outputs, axis=0)
        return new_val, new_val  # (new carry, auxiliary output if needed)
    final_val, intermediates = lax.scan(body_fn, x, jnp.arange(iterations))
    return final_val, intermediates

# -------------------------------------------------------------------------
# 4) Wrap the function with pmap, marking static parameters.
# -------------------------------------------------------------------------
p_process_with_branches_scan = pmap(
    process_with_branches_scan,
    in_axes=(0, None, None, None, None),
    static_broadcasted_argnums=(1, 2, 3, 4)  # iterations, num_branches, branch_depth, scale_factor
)

# -------------------------------------------------------------------------
# 5) Benchmarking function using pmap.
# -------------------------------------------------------------------------
def run_benchmarks_pmap(batch_input, depths=(1_000_000,), num_branches=2, branch_depth=OPTIMAL_DEPTH_STEP,
                        scale_factor=1.0, num_trials=1):
    results = []
    for depth in depths:
        iterations = depth // branch_depth  # For 1M depth, iterations = 4
        print(f"\n--- Running benchmark for total_depth={depth} (iterations={iterations}) ---")
        # Split the input batch into shards for each device.
        sharded_input = batch_input.reshape((NUM_DEVICES, -1))
        for trial in range(num_trials):
            start_time = time.time()
            final_shard, _ = p_process_with_branches_scan(
                sharded_input, iterations, num_branches, branch_depth, scale_factor
            )
            # Reassemble results from devices.
            final_x = jax.device_get(final_shard).reshape(-1)
            jax.block_until_ready(final_x)
            elapsed = time.time() - start_time
            mean_val = float(jnp.mean(final_x))
            results.append({"depth": depth, "trial": trial, "time": elapsed, "mean_output": mean_val})
            print(f"Trial {trial}: Final mean(x)={mean_val:.6f}, total time: {elapsed:.6f} sec")
    return results

# -------------------------------------------------------------------------
# 6) Main Script.
# -------------------------------------------------------------------------
if __name__ == "__main__":
    # Create a batch of 50M samples.
    batch_input = jnp.linspace(0, 10, BATCH_SIZE)

    bench_results = run_benchmarks_pmap(
        batch_input,
        depths=[1_000_000],  # Total depth of 1M, so 4 iterations.
        num_branches=2,
        branch_depth=OPTIMAL_DEPTH_STEP,
        scale_factor=1.0,
        num_trials=1  # Adjust as needed.
    )

    print("\nCollected Benchmark Results:")
    for r in bench_results:
        print(r)



--- Running benchmark for total_depth=1000000 (iterations=4) ---
Trial 0: Final mean(x)=0.004920, total time: 524.522123 sec

Collected Benchmark Results:
{'depth': 1000000, 'trial': 0, 'time': 524.5221230983734, 'mean_output': 0.004919611383229494}


In [ ]:
import jax
import jax.numpy as jnp
from jax import lax, jit, vmap, pmap
from functools import partial
import time

# --- Configuration
MAX_RECURSION_DEPTH    = 1_000_000      # Maximum recursion depth.
OPTIMAL_DEPTH_STEP     = 500_000        # Increased branch step (from 250K to 500K)
DIMENSIONAL_CONSTRAINT = 0.8
BATCH_SIZE             = 50_000_000     # 50M samples
NUM_DEVICES            = jax.local_device_count()  # e.g. 8 cores
LOCAL_BATCH_SIZE       = BATCH_SIZE // NUM_DEVICES

VAL_CLAMP_LOW  = -100.0
VAL_CLAMP_HIGH =  100.0

# -------------------------------------------------------------------------
# 1) Dynamic pi & phi functions (with reduced decay as before)
# -------------------------------------------------------------------------
@jit
def dynamic_pi(depth, scale_factor):
    depth = jnp.minimum(depth, MAX_RECURSION_DEPTH)
    return jnp.pi * jnp.log1p(depth + 1) * scale_factor * DIMENSIONAL_CONSTRAINT

@jit
def dynamic_phi(depth, scale_factor):
    depth = jnp.minimum(depth, MAX_RECURSION_DEPTH)
    # Slow the decay by increasing the denominator factor.
    return (1 + jnp.sqrt(5)) / 2 * jnp.exp(-depth / ((scale_factor * 20) + 1)) * DIMENSIONAL_CONSTRAINT

@jit
def stabilize_depth(depth):
    return depth / (1 + jnp.log1p(depth + 1))

# -------------------------------------------------------------------------
# 2) Base function: linear recursion over a given branch depth.
# -------------------------------------------------------------------------
@partial(jit, static_argnames=["depth"])
def dppu_with_dynamic_pi_phi(x, depth=OPTIMAL_DEPTH_STEP, scale_factor=1.0):
    depth = stabilize_depth(jnp.minimum(depth, MAX_RECURSION_DEPTH))
    def body_fn(i, val):
        pi_dyn  = dynamic_pi(i, scale_factor)
        phi_dyn = dynamic_phi(i, scale_factor)
        scale   = jnp.log1p(i + 1) * scale_factor * DIMENSIONAL_CONSTRAINT
        safe_val = jnp.clip(val, VAL_CLAMP_LOW, VAL_CLAMP_HIGH)
        # Reduce the damping effect slightly.
        new_val = jnp.sin(safe_val * scale * pi_dyn) * jnp.exp(-safe_val / (phi_dyn + 10))
        return new_val
    return lax.fori_loop(0, depth.astype(jnp.int32), body_fn, x)

# -------------------------------------------------------------------------
# 3) Process with Branch Recycling fused with lax.scan.
#    Each device processes its local shard (of shape [LOCAL_BATCH_SIZE])
#    over a number of iterations.
# -------------------------------------------------------------------------
def process_with_branches_scan(x, iterations, num_branches=2, branch_depth=OPTIMAL_DEPTH_STEP, scale_factor=1.0):
    def body_fn(carry, _):
        # Replicate the local shard into branches: shape (num_branches, LOCAL_BATCH_SIZE)
        xs = jnp.stack([carry] * num_branches, axis=0)
        branch_fn = vmap(lambda xi: dppu_with_dynamic_pi_phi(xi, depth=branch_depth, scale_factor=scale_factor))
        branch_outputs = branch_fn(xs)
        new_val = jnp.sum(branch_outputs, axis=0)
        return new_val, new_val  # (new carry, auxiliary output if needed)
    final_val, intermediates = lax.scan(body_fn, x, jnp.arange(iterations))
    return final_val, intermediates

# -------------------------------------------------------------------------
# 4) Wrap the function with pmap, marking static parameters.
# -------------------------------------------------------------------------
p_process_with_branches_scan = pmap(
    process_with_branches_scan,
    in_axes=(0, None, None, None, None),
    static_broadcasted_argnums=(1, 2, 3, 4)  # iterations, num_branches, branch_depth, scale_factor
)

# -------------------------------------------------------------------------
# 5) Benchmarking function using pmap.
# -------------------------------------------------------------------------
def run_benchmarks_pmap(batch_input, depths=(1_000_000,), num_branches=2, branch_depth=OPTIMAL_DEPTH_STEP,
                        scale_factor=1.0, num_trials=1):
    results = []
    for depth in depths:
        iterations = depth // branch_depth  # For 1M depth with 500K steps, iterations = 2
        print(f"\n--- Running benchmark for total_depth={depth} (iterations={iterations}) ---")
        # Split the input batch into shards for each device.
        sharded_input = batch_input.reshape((NUM_DEVICES, -1))
        for trial in range(num_trials):
            start_time = time.time()
            final_shard, _ = p_process_with_branches_scan(
                sharded_input, iterations, num_branches, branch_depth, scale_factor
            )
            # Reassemble results from devices.
            final_x = jax.device_get(final_shard).reshape(-1)
            jax.block_until_ready(final_x)
            elapsed = time.time() - start_time
            mean_val = float(jnp.mean(final_x))
            results.append({"depth": depth, "trial": trial, "time": elapsed, "mean_output": mean_val})
            print(f"Trial {trial}: Final mean(x)={mean_val:.6f}, total time: {elapsed:.6f} sec")
    return results

# -------------------------------------------------------------------------
# 6) Main Script.
# -------------------------------------------------------------------------
if __name__ == "__main__":
    # Create a batch of 50M samples.
    batch_input = jnp.linspace(0, 10, BATCH_SIZE)

    bench_results = run_benchmarks_pmap(
        batch_input,
        depths=[1_000_000],  # Total depth of 1M; with branch_depth=500K, iterations = 2.
        num_branches=2,
        branch_depth=OPTIMAL_DEPTH_STEP,
        scale_factor=1.0,
        num_trials=1  # Adjust as needed.
    )

    print("\nCollected Benchmark Results:")
    for r in bench_results:
        print(r)


--- Running benchmark for total_depth=1000000 (iterations=2) ---
Trial 0: Final mean(x)=0.063186, total time: 498.865307 sec

Collected Benchmark Results:
{'depth': 1000000, 'trial': 0, 'time': 498.86530685424805, 'mean_output': 0.06318572908639908}


In [ ]:
import jax
import jax.numpy as jnp
from jax import lax, jit, vmap, pmap
from functools import partial
import time

# --- Configuration
MAX_RECURSION_DEPTH    = 1_000_000
# We'll use a branch depth of 500K, so with total_depth = 1M we have 2 iterations.
OPTIMAL_DEPTH_STEP     = 500_000
DIMENSIONAL_CONSTRAINT = 0.8
BATCH_SIZE             = 50_000_000     # 50M samples
NUM_DEVICES            = 8              # Adjust as needed
LOCAL_BATCH_SIZE       = BATCH_SIZE // NUM_DEVICES

VAL_CLAMP_LOW  = -100.0
VAL_CLAMP_HIGH =  100.0

@jit
def dynamic_pi(depth, scale_factor):
    depth = jnp.minimum(depth, MAX_RECURSION_DEPTH)
    return jnp.pi * jnp.log1p(depth + 1) * scale_factor * DIMENSIONAL_CONSTRAINT

@jit
def dynamic_phi(depth, scale_factor):
    depth = jnp.minimum(depth, MAX_RECURSION_DEPTH)
    return (1 + jnp.sqrt(5)) / 2 * jnp.exp(-depth / ((scale_factor * 20) + 1)) * DIMENSIONAL_CONSTRAINT

@jit
def stabilize_depth(depth):
    return depth / (1 + jnp.log1p(depth + 1))

@partial(jit, static_argnames=["depth"])
def dppu_with_dynamic_pi_phi(x, depth=OPTIMAL_DEPTH_STEP, scale_factor=1.0):
    depth = stabilize_depth(jnp.minimum(depth, MAX_RECURSION_DEPTH))
    def body_fn(i, val):
        pi_dyn  = dynamic_pi(i, scale_factor)
        phi_dyn = dynamic_phi(i, scale_factor)
        scale   = jnp.log1p(i + 1) * scale_factor * DIMENSIONAL_CONSTRAINT
        safe_val = jnp.clip(val, VAL_CLAMP_LOW, VAL_CLAMP_HIGH)
        new_val = jnp.sin(safe_val * scale * pi_dyn) * jnp.exp(-safe_val / (phi_dyn + 10))
        return new_val
    return lax.fori_loop(0, depth.astype(jnp.int32), body_fn, x)

def branch_recycle(x, num_branches=2, branch_depth=OPTIMAL_DEPTH_STEP, scale_factor=1.0):
    xs = jnp.stack([x] * num_branches, axis=0)
    branch_fn = vmap(lambda xi: dppu_with_dynamic_pi_phi(xi, depth=branch_depth, scale_factor=scale_factor))
    branch_outputs = branch_fn(xs)
    return jnp.sum(branch_outputs, axis=0)

# Mark iterations (and other parameters) as static so that jnp.arange works correctly.
@partial(pmap,
         in_axes=(0, None, None, None, None),
         static_broadcasted_argnums=(1, 2, 3, 4))
def process_with_branches(x, iterations, num_branches, branch_depth, scale_factor):
    def body_fn(carry, _):
        new_val = branch_recycle(carry, num_branches, branch_depth, scale_factor)
        return new_val, new_val
    final_val, _ = lax.scan(body_fn, x, jnp.arange(iterations))
    return final_val

def run_benchmark(batch_input, total_depth=1_000_000, num_branches=2,
                  branch_depth=OPTIMAL_DEPTH_STEP, scale_factor=1.0):
    iterations = total_depth // branch_depth  # For 1M total_depth with 500K branch_depth, iterations = 2.
    sharded_input = batch_input.reshape((NUM_DEVICES, -1))
    start_time = time.time()
    final_shard = process_with_branches(sharded_input, iterations, num_branches, branch_depth, scale_factor)
    final_output = jax.device_get(final_shard).reshape(-1)
    jax.block_until_ready(final_output)
    elapsed = time.time() - start_time
    mean_val = float(jnp.mean(final_output))
    return elapsed, mean_val

if __name__ == "__main__":
    # Create a 50M-sample input.
    batch_input = jnp.linspace(0, 10, BATCH_SIZE)
    elapsed, mean_val = run_benchmark(batch_input)
    print(f"Time taken: {elapsed:.6f} sec")
    print(f"Mean output: {mean_val:.6f}")




Time taken: 500.216080 sec
Mean output: 0.063186
